# GPT-CUDA Minimal Test v2 (EMBEDDING BUG FIXED)

In [ ]:
!nvidia-smi

In [ ]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt
!wc -c input.txt
print('OK')

In [ ]:
%%writefile Layer.cuh
#pragma once
#include <cublas_v2.h>

#define ACTIVATION_NONE 0
#define ACTIVATION_RELU 1
#define ACTIVATION_GELU 2

class Layer {
  protected:
    int batch_features, in_features, out_features;
  public:
    Layer(int bs, int in_f, int out_f) : batch_features(bs), in_features(in_f), out_features(out_f) {}
    virtual ~Layer() {}
    virtual float* forward(cublasHandle_t h, void* inp, int act) = 0;
    virtual float* backward(cublasHandle_t h, float* dY) = 0;
    virtual void step(float lr, int t) = 0;
};


In [ ]:
%%writefile LinearLayer.cuh
#pragma once
#include "Layer.cuh"
class LinearLayer : public Layer {
  float *d_W, *d_b, *d_dW, *d_db, *d_X_cache, *d_Y, *d_dX, *d_m, *d_v;
  float *d_pre_act;  // saved pre-activation for backward
  float *d_b_m, *d_b_v;  // Adam state for bias
  int last_activation;    // which activation was applied (NONE/RELU/GELU)
public:
  LinearLayer(int bs, int in_f, int out_f);
  ~LinearLayer();
  float* forward(cublasHandle_t h, void* inp, int act) override;
  float* backward(cublasHandle_t h, float* dY) override;
  void step(float lr, int t) override;
};


In [ ]:
%%writefile LinearLayer.cu
#include "LinearLayer.cuh"
#include <cmath>
#include <cstdlib>

// ─── Xavier/Glorot uniform initialization ──────────────────────────────
__global__ void xavier_init_kernel(float* W, int in_f, int out_f, unsigned int seed) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = in_f * out_f;
    if (idx < total) {
        float limit = sqrtf(6.0f / (float)(in_f + out_f));
        // Simple pseudo-random on GPU using thread index + seed
        unsigned int s = idx + seed;
        s = (s ^ 61) ^ (s >> 16);
        s = s * 9;
        s = s ^ (s >> 4);
        s = s * 0x27d4eb2d;
        s = s ^ (s >> 15);
        float r = (float)(s & 0xFFFF) / 65535.0f;
        W[idx] = (2.0f * r - 1.0f) * limit;
    }
}

// ─── FIX #1: ReLU activation kernel ────────────────────────────────────
__global__ void relu_forward_kernel(float* d_Y, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        if (d_Y[idx] < 0.0f) d_Y[idx] = 0.0f;
    }
}

__global__ void relu_backward_kernel(float* d_dY, float* d_Y, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        if (d_Y[idx] <= 0.0f) d_dY[idx] = 0.0f;
    }
}

// ─── FIX #5: GELU activation kernel (approximation) ────────────────────
__global__ void gelu_forward_kernel(float* d_Y, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        float x = d_Y[idx];
        float cdf = 0.5f * (1.0f + tanhf(0.7978845608f * (x + 0.044715f * x * x * x)));
        d_Y[idx] = x * cdf;
    }
}

__global__ void gelu_backward_kernel(float* d_dY, float* d_Y_save, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        float x = d_Y_save[idx];
        float x3 = x * x * x;
        float tanh_arg = 0.7978845608f * (x + 0.044715f * x3);
        float tanh_val = tanhf(tanh_arg);
        float sech2 = 1.0f - tanh_val * tanh_val;
        float gelu_deriv = 0.5f * (1.0f + tanh_val) + 0.5f * x * 0.7978845608f * (1.0f + 3.0f * 0.044715f * x * x) * sech2;
        d_dY[idx] *= gelu_deriv;
    }
}


// ─── Bias gradient kernel ──────────────────────────────────────────────
__global__ void bias_grad_kernel(float* d_db, float* d_dY, int batch_f, int out_f) {
    int tid = threadIdx.x;
    extern __shared__ float shared_sum[];
    shared_sum[tid] = 0.0f;
    for (int b = 0; b < batch_f; b++) {
        shared_sum[tid] += d_dY[b * out_f + tid];
    }
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1) {
        if (tid < s) shared_sum[tid] += shared_sum[tid + s];
        __syncthreads();
    }
    if (tid == 0) d_db[blockIdx.x * blockDim.x + tid] = shared_sum[0];
}


// ─── Bias Adam kernel (1D) ─────────────────────────────────────────────
__global__ void bias_adam_kernel(float* b, float* db, float* m, float* v, float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float b1 = 0.9f, b2 = 0.999f, eps = 1e-8f;
        m[idx] = b1 * m[idx] + (1.0f - b1) * db[idx];
        v[idx] = b2 * v[idx] + (1.0f - b2) * (db[idx] * db[idx]);
        float m_hat = m[idx] / (1.0f - powf(b1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(b2, (float)t));
        b[idx] -= lr * m_hat / (sqrtf(v_hat) + eps);
        db[idx] = 0.0f;
    }
}

// ─── AdamW kernel ───────────────────────────────────────────────────────
__global__ void linear_adam_kernel(float* W, float* dW, float* m, float* v, float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float b1 = 0.9f, b2 = 0.999f, eps = 1e-8f, wd = 0.01f;
        m[idx] = b1 * m[idx] + (1.0f - b1) * dW[idx];
        v[idx] = b2 * v[idx] + (1.0f - b2) * (dW[idx] * dW[idx]);
        float m_hat = m[idx] / (1.0f - powf(b1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(b2, (float)t));
        W[idx] -= lr * wd * W[idx];
        W[idx] -= lr * m_hat / (sqrtf(v_hat) + eps);
        dW[idx] = 0.0f;
    }
}

__global__ void add_bias_kernel(float* Y, const float* b, int batch_f, int out_f) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < batch_f * out_f) {
        Y[idx] += b[idx % out_f];
    }
}

// ─── Constructor with Xavier init ───────────────────────────────────────
LinearLayer::LinearLayer(int bs, int in_f, int out_f) : Layer(bs, in_f, out_f) {
    cudaMalloc(&d_W, sizeof(float) * in_f * out_f);
    cudaMalloc(&d_b, sizeof(float) * out_f);
    cudaMalloc(&d_dW, sizeof(float) * in_f * out_f);
    cudaMalloc(&d_db, sizeof(float) * out_f);
    cudaMalloc(&d_Y, sizeof(float) * bs * out_f);
    cudaMalloc(&d_pre_act, sizeof(float) * bs * out_f);
    cudaMalloc(&d_dX, sizeof(float) * bs * in_f);

    // FIX #3: Xavier/Glorot uniform initialization
    int threads = 256;
    int blocks = (in_f * out_f + threads - 1) / threads;
    xavier_init_kernel<<<blocks, threads>>>(d_W, in_f, out_f, (unsigned int)time(0) + in_f * 31 + out_f * 17);
    cudaMemset(d_b, 0, sizeof(float) * out_f);

    cudaMalloc(&d_m, sizeof(float) * in_f * out_f);
    cudaMalloc(&d_v, sizeof(float) * in_f * out_f);
    cudaMemset(d_m, 0, sizeof(float) * in_f * out_f);
    cudaMemset(d_v, 0, sizeof(float) * in_f * out_f);

    cudaMalloc(&d_b_m, sizeof(float) * out_f);
    cudaMalloc(&d_b_v, sizeof(float) * out_f);
    cudaMemset(d_b_m, 0, sizeof(float) * out_f);
    cudaMemset(d_b_v, 0, sizeof(float) * out_f);
    last_activation = ACTIVATION_NONE;
}

LinearLayer::~LinearLayer() {
    cudaFree(d_W); cudaFree(d_b); cudaFree(d_dW); cudaFree(d_db);
    cudaFree(d_Y); cudaFree(d_pre_act); cudaFree(d_dX); cudaFree(d_m); cudaFree(d_v);
    cudaFree(d_b_m); cudaFree(d_b_v);
}

float* LinearLayer::forward(cublasHandle_t h, void* inp, int act) {
    d_X_cache = (float*)inp;
    const float alpha = 1.0f, beta = 0.0f;

    cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N,
                out_features, batch_features, in_features,
                &alpha, d_W, out_features, d_X_cache, in_features,
                &beta, d_Y, out_features);

    int threads = 256;
    int blocks_bias = (batch_features * out_features + threads - 1) / threads;
    add_bias_kernel<<<blocks_bias, threads>>>(d_Y, d_b, batch_features, out_features);

    // Save pre-activation for backward pass (only if activation will be applied)
    last_activation = act;
    if (act != ACTIVATION_NONE) {
        int total_act = batch_features * out_features;
        cudaMemcpy(d_pre_act, d_Y, sizeof(float) * total_act, cudaMemcpyDeviceToDevice);
        int blocks_act = (total_act + threads - 1) / threads;
        if (act == ACTIVATION_RELU) {
            relu_forward_kernel<<<blocks_act, threads>>>(d_Y, total_act);
        } else if (act == ACTIVATION_GELU) {
            gelu_forward_kernel<<<blocks_act, threads>>>(d_Y, total_act);
        }
    }

    return d_Y;
}

float* LinearLayer::backward(cublasHandle_t h, float* d_dY) {
    const float alpha = 1.0f, beta = 0.0f;
    int total = batch_features * out_features;
    int threads = 256;

    // FIX #8: Apply activation backward (GELU/ReLU derivative)
    if (last_activation == ACTIVATION_RELU) {
        int blocks_act = (total + threads - 1) / threads;
        relu_backward_kernel<<<blocks_act, threads>>>(d_dY, d_pre_act, total);
    } else if (last_activation == ACTIVATION_GELU) {
        int blocks_act = (total + threads - 1) / threads;
        gelu_backward_kernel<<<blocks_act, threads>>>(d_dY, d_pre_act, total);
    }

    // Compute bias gradient: d_db = sum(d_dY over batch)
    cudaMemset(d_db, 0, sizeof(float) * out_features);
    int blocks_bias = (out_features + threads - 1) / threads;
    bias_grad_kernel<<<blocks_bias, threads, threads * sizeof(float)>>>(d_db, d_dY, batch_features, out_features);

    // dX = dY * W^T
    cublasSgemm(h, CUBLAS_OP_T, CUBLAS_OP_N,
                in_features, batch_features, out_features,
                &alpha, d_W, out_features, d_dY, out_features,
                &beta, d_dX, in_features);

    // dW = X^T * dY
    cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_T,
                out_features, in_features, batch_features,
                &alpha, d_dY, out_features, d_X_cache, in_features,
                &beta, d_dW, out_features);

    return d_dX;
}

void LinearLayer::step(float lr, int t) {
    int total = in_features * out_features;
    int threads = 256;
    int blocks = (total + threads - 1) / threads;
    linear_adam_kernel<<<blocks, threads>>>(d_W, d_dW, d_m, d_v, lr, t, total);

    // FIX #9: Also update bias with Adam
    int blocks_b = (out_features + threads - 1) / threads;
    bias_adam_kernel<<<blocks_b, threads>>>(d_b, d_db, d_b_m, d_b_v, lr, t, out_features);
}


In [ ]:
%%writefile EmbeddingLayer.cuh
#pragma once
#include "Layer.cuh"

class EmbeddingLayer : public Layer {
  private:
    // Poids et gradients
    float* d_W;  // Le dictionnaire [vocab_size * embedding_dim]
    float* d_dW; // Le gradient des poids

    // Encodage Positionnel (Constant)
    float* d_PE; // La matrice des ondes [context_size * embedding_dim]
    
    // Entrées / Sorties
    int* d_X;    // L'entrée (Tableau d'entiers) [batch_size * context_size]
    float* d_Y;  // La sortie [batch_size * context_size * embedding_dim]
    
    // Les dimensions
    int vocab_size;
    int embedding_dim;
    int batch_size;
    int context_size;

    //ADAM
    float* d_m;
    float* d_v;
    
  public:
    EmbeddingLayer(int vocab_size, int embedding_dim, int batch_size, int context_size);
    ~EmbeddingLayer();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate,int t) override;
};


In [ ]:
%%writefile EmbeddingLayer.cu
#include "EmbeddingLayer.cuh"
#include <cmath>
#include <cstdlib>

// Re-use the xavier init kernel (declare extern)
__global__ void xavier_init_kernel(float* W, int in_f, int out_f, unsigned int seed);

__global__ void embedding_forward_kernel(int* X, float* W, float* PE, float* Y, int emb_dim, int cs, int total) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if (idx < total) {
        int e_idx = idx % emb_dim;
        int w_pos = idx / emb_dim;
        int s_pos = w_pos % cs;
        int v_id = X[w_pos];
        Y[idx] = W[v_id * emb_dim + e_idx] + PE[s_pos * emb_dim + e_idx];
    }
}

__global__ void backward_embedding_kernel(int* X, float* dY, float* dW, int emb_dim, int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        int e_idx = idx % emb_dim;
        int w_pos = idx / emb_dim;
        int v_id = X[w_pos];
        atomicAdd(&dW[v_id * emb_dim + e_idx], dY[idx]);
    }
}

__global__ void embedding_adam_kernel(float* W, float* dW, float* m, float* v, float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float b1 = 0.9f, b2 = 0.999f, eps = 1e-8f, wd = 0.01f;
        m[idx] = b1 * m[idx] + (1.0f - b1) * dW[idx];
        v[idx] = b2 * v[idx] + (1.0f - b2) * (dW[idx] * dW[idx]);
        float m_hat = m[idx] / (1.0f - powf(b1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(b2, (float)t));
        W[idx] -= lr * wd * W[idx];
        W[idx] -= lr * m_hat / (sqrtf(v_hat) + eps);
        dW[idx] = 0.0f;
    }
}

EmbeddingLayer::EmbeddingLayer(int vs, int ed, int bs, int cs) : Layer(bs, cs, ed) {
    vocab_size = vs; embedding_dim = ed; batch_size = bs; context_size = cs;
    cudaMalloc(&d_W, sizeof(float) * vs * ed);
    cudaMalloc(&d_dW, sizeof(float) * vs * ed);
    cudaMalloc(&d_Y, sizeof(float) * bs * cs * ed);
    cudaMalloc(&d_PE, sizeof(float) * cs * ed);

    // FIX #3: Xavier init on GPU
    int threads = 256;
    int blocks = (vs * ed + threads - 1) / threads;
    xavier_init_kernel<<<blocks, threads>>>(d_W, vs, ed, 42);

    // Positional encoding on CPU
    float* h_PE = (float*)malloc(sizeof(float) * cs * ed);
    for (int pos = 0; pos < cs; pos++) {
        for (int i = 0; i < ed; i += 2) {
            float div = pow(10000.0f, (float)i / ed);
            h_PE[pos * ed + i] = sin(pos / div);
            if (i + 1 < ed) h_PE[pos * ed + i + 1] = cos(pos / div);
        }
    }
    cudaMemcpy(d_PE, h_PE, sizeof(float) * cs * ed, cudaMemcpyHostToDevice);
    free(h_PE);

    cudaMalloc(&d_m, sizeof(float) * vs * ed);
    cudaMalloc(&d_v, sizeof(float) * vs * ed);
    cudaMemset(d_m, 0, sizeof(float) * vs * ed);
    cudaMemset(d_v, 0, sizeof(float) * vs * ed);
}

EmbeddingLayer::~EmbeddingLayer() {
    cudaFree(d_W); cudaFree(d_dW); cudaFree(d_Y); cudaFree(d_PE); cudaFree(d_m); cudaFree(d_v);
}

float* EmbeddingLayer::forward(cublasHandle_t h, void* inp, int act) {
    d_X = (int*)inp;
    int total = batch_size * context_size * embedding_dim;  // ALL output elements!
    int tpb = 256;
    int bpg = (total + tpb - 1) / tpb;
    embedding_forward_kernel<<<bpg, tpb>>>(d_X, d_W, d_PE, d_Y, embedding_dim, context_size, total);
    cudaDeviceSynchronize();
    return d_Y;
}

float* EmbeddingLayer::backward(cublasHandle_t h, float* d_dY) {
    int total = batch_size * context_size * embedding_dim;
    int tpb = 256;
    int bpg = (total + tpb - 1) / tpb;
    cudaMemset(d_dW, 0, sizeof(float) * vocab_size * embedding_dim);
    backward_embedding_kernel<<<bpg, tpb>>>(d_X, d_dY, d_dW, embedding_dim, total);
    cudaDeviceSynchronize();
    return nullptr;
}

void EmbeddingLayer::step(float lr, int t) {
    int total = vocab_size * embedding_dim;
    int threads = 256;
    int blocks = (total + threads - 1) / threads;
    embedding_adam_kernel<<<blocks, threads>>>(d_W, d_dW, d_m, d_v, lr, t, total);
}


In [ ]:
%%writefile DataLoader.h
#pragma once
#include <string>
#include <vector>
#include <map>
class DataLoader {
  std::string raw_text; std::vector<int> tokens;
  std::map<char,int> c2i; std::map<int,char> i2c;
  int bs, cs, vs;
public:
  DataLoader(const std::string& fp, int bs, int cs);
  ~DataLoader();
  void get_batch(int* X, int* Y);
  int get_vocab_size() const { return vs; }
  int get_num_tokens() const { return (int)tokens.size(); }
  std::map<int,char> get_i2c() const { return i2c; }
};


In [ ]:
%%writefile DataLoader.cpp
#include "DataLoader.h"
#include <fstream>
#include <sstream>
#include <iostream>
#include <set>
#include <cstdlib>

DataLoader::DataLoader(const std::string& fp, int bs, int cs) {
    this->bs = bs; this->cs = cs;
    std::ifstream file(fp);
    if (!file.is_open()) { std::cerr << "ERROR: cannot open " << fp << std::endl; exit(1); }
    std::stringstream buf; buf << file.rdbuf(); raw_text = buf.str(); file.close();
    std::set<char> uniq(raw_text.begin(), raw_text.end());
    vs = uniq.size();
    int i = 0;
    for (char c : uniq) { c2i[c] = i; i2c[i] = c; i++; }
    tokens.reserve(raw_text.size());
    for (char c : raw_text) tokens.push_back(c2i[c]);
    std::cout << "DataLoader: " << tokens.size() << " chars, vocab=" << vs << std::endl;
}
DataLoader::~DataLoader() {}
void DataLoader::get_batch(int* X, int* Y) {
    for (int b = 0; b < bs; b++) {
        int start = rand() % (tokens.size() - cs - 1);
        for (int i = 0; i < cs; i++) {
            X[b * cs + i] = tokens[start + i];
            Y[b * cs + i] = tokens[start + i + 1];
        }
    }
}


In [ ]:
%%writefile main_minimal.cu
/**
 * GPT-CUDA MINIMAL: Bigram model (embedding + linear head, NO transformer blocks)
 * Diagnostic test: if this learns, the bug is in the transformer blocks.
 * If this doesn't learn, the bug is in the training loop.
 */
#include <iostream>
#include <cublas_v2.h>
#include "EmbeddingLayer.cuh"
#include "LinearLayer.cuh"
#include "DataLoader.h"
#include <vector>
#include <string>
#include <cmath>
#include <fstream>
#include <ctime>

__global__ void cross_entropy_backward_kernel(float* logits, int* targets, float* dY,
                                               int vs, int tw) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < tw) {
        int tgt = targets[idx];
        float mx = -1e9f;
        for (int i = 0; i < vs; i++) mx = fmaxf(mx, logits[idx * vs + i]);
        float sum = 0.0f;
        for (int i = 0; i < vs; i++) sum += expf(logits[idx * vs + i] - mx);
        for (int i = 0; i < vs; i++) {
            float p = expf(logits[idx * vs + i] - mx) / (sum + 1e-7f);
            dY[idx * vs + i] = (p - (i == tgt ? 1.0f : 0.0f)) / (float)tw;
        }
    }
}

__global__ void clip_gradients_kernel(float* dY, float mn, float mx, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        float v = dY[idx];
        if (v > mx) v = mx; if (v < mn) v = mn;
        if (isnan(v)) v = 0.0f;
        dY[idx] = v;
    }
}

int main() {
    cublasHandle_t h; cublasCreate(&h);

    int cs = 64, bs = 64, ed = 192;
    float base_lr = 3e-3f;
    int total_iter = 5000, warmup = 1000, log_every = 50;

    DataLoader dl("input.txt", bs, cs);
    int vs = dl.get_vocab_size();
    int tw = bs * cs;

    std::cout << "=== GPT-CUDA MINIMAL (Bigram: Embedding + LM Head only) ===\n";
    std::cout << "Vocab: " << vs << ", Emb: " << ed << ", Tokens/batch: " << tw << "\n";

    // Minimal model: embedding + linear head (no transformer blocks)
    EmbeddingLayer emb(vs, ed, bs, cs);
    LinearLayer head(bs * cs, ed, vs);

    int *hX = (int*)malloc(sizeof(int)*tw), *hT = (int*)malloc(sizeof(int)*tw);
    int *dX, *dT; float *d_dY;
    cudaMalloc(&dX, sizeof(int)*tw);
    cudaMalloc(&dT, sizeof(int)*tw);
    cudaMalloc(&d_dY, sizeof(float)*tw*vs);

    std::ofstream log("training_log_minimal.csv");
    log << "iteration,loss,lr\n";
    time_t t0 = time(0);

    for (int iter = 0; iter < total_iter; iter++) {
        float lr;
        if (iter < warmup)
            lr = base_lr * ((float)(iter+1)/(float)warmup);
        else {
            float p = (float)(iter-warmup)/(float)(total_iter-warmup);
            lr = base_lr * (0.1f + 0.45f*(1.0f+cosf(3.14159265f*p)));
        }

        dl.get_batch(hX, hT);
        cudaMemcpy(dX, hX, sizeof(int)*tw, cudaMemcpyHostToDevice);
        cudaMemcpy(dT, hT, sizeof(int)*tw, cudaMemcpyHostToDevice);

        float* d_emb = emb.forward(h, dX, 0);
        float* d_logits = head.forward(h, d_emb, 0);

        if (iter % log_every == 0) {
            float* h_logits = (float*)malloc(sizeof(float)*tw*vs);
            cudaMemcpy(h_logits, d_logits, sizeof(float)*tw*vs, cudaMemcpyDeviceToHost);
            float loss = 0.0f;
            for (int i = 0; i < tw; i++) {
                int tgt = hT[i];
                float mx = -1e9f;
                for (int v = 0; v < vs; v++) mx = std::max(mx, h_logits[i*vs+v]);
                float sum = 0.0f;
                for (int v = 0; v < vs; v++) sum += expf(h_logits[i*vs+v] - mx);
                loss += -logf(expf(h_logits[i*vs+tgt]-mx)/sum + 1e-7f);
            }
            loss /= tw;
            free(h_logits);
            std::cout << "[" << iter << "/" << total_iter << "] loss=" << loss
                      << " lr=" << lr << " time=" << (time(0)-t0) << "s\n";
            log << iter << "," << loss << "," << lr << "\n";
        }

        int th = 256;
        int bce = (tw + th - 1) / th;
        cross_entropy_backward_kernel<<<bce, th>>>(d_logits, dT, d_dY, vs, tw);
        int tgrad = tw * vs;
        int bclip = (tgrad + th - 1) / th;
        clip_gradients_kernel<<<bclip, th>>>(d_dY, -5.0f, 5.0f, tgrad);
        cudaDeviceSynchronize();

        float* d_grad = head.backward(h, d_dY);
        emb.backward(h, d_grad);
        head.step(lr, iter+1);
        emb.step(lr, iter+1);
    }

    log.close();
    std::cout << "\n=== DONE ===\n";

    free(hX); free(hT);
    cudaFree(dX); cudaFree(dT); cudaFree(d_dY);
    cublasDestroy(h);
    return 0;
}


In [ ]:
!nvcc -O3 -o gpt_minimal main_minimal.cu EmbeddingLayer.cu LinearLayer.cu DataLoader.cpp -lcublas -Wno-deprecated-gpu-targets 2>&1
import os; assert os.path.exists('gpt_minimal'), 'FAIL'
print('OK')

In [ ]:
!./gpt_minimal

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df=pd.read_csv('training_log_minimal.csv')
fig,(a,b)=plt.subplots(1,2,figsize=(14,4))
a.plot(df['iteration'],df['loss']); a.set_xlabel('Iter'); a.set_ylabel('Loss'); a.set_title('Minimal Bigram Loss'); a.grid(True,alpha=.3)
b.plot(df['iteration'],df['lr']); b.set_xlabel('Iter'); b.set_ylabel('LR'); b.set_title('LR'); b.grid(True,alpha=.3)
plt.tight_layout(); plt.savefig('minimal_curve.png',dpi=100,bbox_inches='tight'); plt.show()